In [4]:
import polars as pl
import seaborn as sns
import datetime as dt

In [5]:
# desativar a notação científica
pl.Config.set_fmt_float("full")
# definir máximo de casas decimais
pl.Config.set_float_precision(2)

polars.config.Config

In [9]:
ARQUIVO = r'C:\Users\Aluno.SantaLuzia\Documents\Elaine\uc2\bases\dados_bronze\df_bf.parquet'

In [10]:
try:
    hora_inicio = dt.datetime.now()
    #criar um plano de execução
    df_bf_plano = pl.scan_parquet(ARQUIVO)
    df_lazy = df_bf_plano.select(
        [pl.col('NOME MUNICÍPIO'),
         pl.col('VALOR PARCELA'),
         pl.col('UF'),
         pl.col('MÊS REFERÊNCIA'),
         pl.col('MÊS COMPETÊNCIA')
        ]).with_columns(
            (pl.col('MÊS REFERÊNCIA')
                .cast(pl.Utf8)
                .str.strptime(pl.Date, format='%Y%m')
            ),
            (pl.col('MÊS COMPETÊNCIA')
                .cast(pl.Utf8)
                .str.strptime(pl.Date, format='%Y%m')
            )
        )
    
    hora_fim = dt.datetime.now()
    print(f"Tempo gasto: {hora_fim - hora_inicio}")

except Exception as e:
    print(f'Erro ao obter dados do parquet: {e}')

Tempo gasto: 0:00:00.000373


In [11]:
COLUNA = 'VALOR PARCELA'

df_analise = df_lazy.select(
    pl.col(COLUNA).mean().alias('Média'),
    pl.col(COLUNA).median().alias('Mediana'),
    pl.col(COLUNA).std().alias('Desvio Padrão'),
    pl.col(COLUNA).skew().alias('Assimetria'),
    pl.col(COLUNA).kurtosis().alias('Curtose'),
    pl.col(COLUNA).min().alias('Mínimo'),
    pl.col(COLUNA).quantile(0.25).alias('Q1'),
    pl.col(COLUNA).quantile(0.75).alias('Q3'),
    pl.col(COLUNA).max().alias('Máximo')
).collect()

df_analise

Média,Mediana,Desvio Padrão,Assimetria,Curtose,Mínimo,Q1,Q3,Máximo
f64,f64,f64,f64,f64,f64,f64,f64,f64
668.26,650.00,190.28,0.89,4.67,25.00,600.00,750.00,3938.00
